# Cabino — Image Generation Evaluation Pipeline

This notebook evaluates the quality of Cabino's AI-generated kitchen visualizations.

**How it works:**
1. Define test cases (room photo + cabinet references)
2. Run generation for each test case using the current prompt
3. Score each output with Gemini Vision against the rubric
4. Review results and compare across prompt versions

**Before running:** Mount your Google Drive and set your Gemini API key in the Config cell.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q google-genai Pillow pandas

In [ ]:
# ── 2. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────────
# @title Configuration { display-mode: "form" }

GEMINI_API_KEY = ""  # @param {type:"string"}

# Path to your test kitchen photos in Google Drive
ROOMS_FOLDER = "/content/drive/MyDrive/Cabinet Examples/test_kitchens"  # @param {type:"string"}

# Path to your cabinet reference photos in Google Drive
CABINETS_FOLDER = "/content/drive/MyDrive/Cabinet Examples/cabinet_refs"  # @param {type:"string"}

# Where to save generated images and results
OUTPUT_FOLDER = "/content/drive/MyDrive/Cabinet Examples/eval_runs"  # @param {type:"string"}

# A label for this run (e.g. 'v1', 'v2-extend-fix') — used to compare runs later
RUN_LABEL = "v1"  # @param {type:"string"}

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print('Config OK')
print(f'Rooms folder exists: {os.path.exists(ROOMS_FOLDER)}')
print(f'Cabinets folder exists: {os.path.exists(CABINETS_FOLDER)}')

In [ ]:
# ── 4. Define Test Cases ─────────────────────────────────────────────────────
# Each test case needs:
#   id          - unique identifier
#   name        - human-readable description
#   room        - filename in ROOMS_FOLDER
#   cabinets    - list of filenames in CABINETS_FOLDER
#   extend      - True if cabinets should go to ceiling
#   stage       - True if room should be staged
#   notes       - anything the scorer should know about this kitchen

TEST_CASES = [
    {
        "id": "tc_001",
        "name": "Brown wood kitchen → white bar handles",
        "room": "kitchen_brown_wood.png",
        "cabinets": ["white_cabinets_bar_handles.jpg"],
        "extend": False,
        "stage": False,
        "notes": "Large style contrast — brown wood to white. Good test of material replacement fidelity."
    },
    {
        "id": "tc_002",
        "name": "Brown wood kitchen → white round handles",
        "room": "kitchen_brown_wood.png",
        "cabinets": ["white_cabinets_round_handles.png"],
        "extend": False,
        "stage": False,
        "notes": "Same kitchen as tc_001 but different handle style. Compare handle rendering accuracy."
    },
    {
        "id": "tc_003",
        "name": "Modern white kitchen → white bar handles (subtle change)",
        "room": "kitchen_modern_white_cabinets.png",
        "cabinets": ["white_cabinets_bar_handles.jpg"],
        "extend": False,
        "stage": False,
        "notes": "Already white cabinets — subtle style change. Tests if model makes unnecessary changes."
    },
    {
        "id": "tc_004",
        "name": "Navy L-shaped → white bar handles (extend to ceiling)",
        "room": "kitchen_navy_lshaped.png",
        "cabinets": ["white_cabinets_bar_handles.jpg"],
        "extend": True,
        "stage": False,
        "notes": "L-shaped layout with extend-to-ceiling enabled. Tests ceiling flush transition and bulkhead removal."
    },
    {
        "id": "tc_005",
        "name": "White kitchen with fridge → white round handles",
        "room": "kitchen_white_fridge.png",
        "cabinets": ["white_cabinets_round_handles.png"],
        "extend": False,
        "stage": False,
        "notes": "Prominent fridge visible. Key test for fridge_unchanged criterion."
    },
    {
        "id": "tc_006",
        "name": "White L-shaped → white bar handles (staged)",
        "room": "kitchen_white_lshaped.png",
        "cabinets": ["white_cabinets_bar_handles.jpg"],
        "extend": False,
        "stage": True,
        "notes": "Staging enabled — countertops should be cleared and re-staged with espresso machine, fruit bowl, cutting board."
    },
    {
        "id": "tc_007",
        "name": "White shaker island → both cabinet refs",
        "room": "kitchen_white_shaker_island.png",
        "cabinets": ["white_cabinets_bar_handles.jpg", "white_cabinets_round_handles.png"],
        "extend": False,
        "stage": False,
        "notes": "Island kitchen with two reference images provided. Tests multi-ref synthesis."
    },
    {
        "id": "tc_008",
        "name": "White shaker island → white bar handles (extend to ceiling)",
        "room": "kitchen_white_shaker_island.png",
        "cabinets": ["white_cabinets_bar_handles.jpg"],
        "extend": True,
        "stage": False,
        "notes": "Island layout with extend-to-ceiling. Island cabinets should NOT extend — only wall uppers."
    },
]

print(f'{len(TEST_CASES)} test cases defined')
for tc in TEST_CASES:
    print(f"  {tc['id']}: {tc['name']}")

In [ ]:
# ── 5. Define Rubric ─────────────────────────────────────────────────────────
# Each criterion:
#   id          - used in results
#   name        - displayed in output
#   description - what Gemini checks (be specific — this is the scoring prompt)
#   weight      - multiplier for weighted score (1.0 = normal, 2.0 = double weight)

RUBRIC = [
    {
        "id": "cabinet_match",
        "name": "Cabinet Match",
        "description": "Do the rendered cabinets closely match the reference cabinet photos in style, door profile, color, and material? Score 5 if nearly identical, 0 if completely different.",
        "weight": 2.0
    },
    {
        "id": "fridge_unchanged",
        "name": "Fridge Unchanged",
        "description": "Is the fridge in the exact same position and does it look the same as in the original photo? Score 0 if the fridge moved, changed style, or disappeared. Score 5 if completely unchanged.",
        "weight": 2.0
    },
    {
        "id": "stove_unchanged",
        "name": "Stove / Range Unchanged",
        "description": "Is the stove/range in the exact same position and unchanged in appearance? Score 0 if it moved or changed, 5 if identical to original.",
        "weight": 2.0
    },
    {
        "id": "sink_unchanged",
        "name": "Sink Unchanged",
        "description": "Is the kitchen sink in the same position and unchanged? Score 0 if it moved or changed significantly, 5 if identical.",
        "weight": 2.0
    },
    {
        "id": "floor_unchanged",
        "name": "Floor Unchanged",
        "description": "Does the floor look identical to the original photo (same material, pattern, color)? Score 0 if floor changed, 5 if identical.",
        "weight": 1.5
    },
    {
        "id": "windows_unchanged",
        "name": "Windows / Doors Unchanged",
        "description": "Are all windows and doors in the same position and unchanged? Score 0 if any window/door moved or changed, 5 if all identical.",
        "weight": 1.5
    },
    {
        "id": "no_hallucinations",
        "name": "No Hallucinations",
        "description": "Did the AI add anything that wasn't there originally or in the reference photos — new decor, clutter, furniture, fixtures, ceiling lights, or items on surfaces? Score 5 if nothing was added, 0 if many things were hallucinated.",
        "weight": 1.5
    },
    {
        "id": "countertop_items",
        "name": "Countertop Items Preserved",
        "description": "Are the items on the countertops the same as in the original photo (same count and position)? Score 5 if identical, 0 if significantly different.",
        "weight": 1.0
    },
    {
        "id": "photorealism",
        "name": "Photorealism",
        "description": "Does the generated image look like a real photograph? Is lighting consistent, are shadows realistic, do materials look real? Score 5 for photographic quality, 0 for clearly AI-generated/fake.",
        "weight": 1.0
    },
    {
        "id": "seamless_integration",
        "name": "Seamless Integration",
        "description": "Do the new cabinets look naturally integrated into the room, or do they look pasted on? Are scale, perspective, and lighting consistent with the rest of the room?",
        "weight": 1.5
    },
]

print(f'{len(RUBRIC)} criteria defined')
for c in RUBRIC:
    print(f"  {c['id']} (weight: {c['weight']})")

In [ ]:
# ── 6. Prompts ───────────────────────────────────────────────────────────────
# These mirror the prompts in src/lib/gemini.ts
# Edit these to test new prompt versions — that's the main thing you'll iterate on

MASTER_PROMPT = """Analyze the provided room photo and the cabinet reference photos. Create a highly detailed, photorealistic image-to-image editing prompt that instructs an AI to replace the existing cabinets in the room with the ones shown in the reference photos. The prompt should specify details about lighting, shadows, scale, material texture, and placement to ensure a seamless integration. At the end of the prompt, include these instructions: Keep the fridge, stove, freezer, and kitchen sink in their exact original spots and maintain their original design. Strictly maintain the original height and layout of the upper cabinets; do not extend them to the ceiling even if the reference photo shows ceiling-height cabinets. Maintain the exact floor rug pattern, floor planks, wall outlets, window details, and ceiling surfaces without alteration. Strictly maintain the exact count and position of all items currently present on the countertops. Additionally, if the upper cabinets do not extend to the ceiling, you must also preserve any items currently resting on top of them. Do strictly not introduce any new bags, clutter, wall decor, floor items, ceiling fixtures, or loose items to any surface. Only if there is existing pendant lighting in the original photo, update its style to match the new aesthetic; do not add new hanging lights if none exist, and do not alter or replace the hood fan, ventilation, or any functional appliances. Change the wall paint and kitchen backsplash tiles to reflect the aesthetic."""

EXTEND_PROMPT = """Analyze the provided room photo and the cabinet reference photos. Create a highly detailed, photorealistic image-to-image editing prompt that instructs an AI to replace the existing cabinets in the room with the ones shown in the reference photos. The prompt should specify details about lighting, shadows, scale, material texture, and placement to ensure a seamless integration. At the end of the prompt, include these instructions: Keep the fridge, stove, freezer, and kitchen sink in their exact original spots and maintain their original design.

Strictly clear any objects from the empty space directly above the cabinetry. All upper cabinetry must now be extended upward to the full height of the ceiling. Use a minimalist, ultra-thin flat scribe filler to transition the cabinet doors to the ceiling surface, ensuring cabinet doors must meet the ceiling surface flush, eliminating any bulkheads, empty space, or shadowy gaps.

Maintain the exact floor rug pattern, floor planks, wall outlets, window details, and ceiling surfaces without alteration. Strictly maintain the exact count and position of all items currently present on the countertops. Do strictly not introduce any new bags, clutter, wall decor, floor items, ceiling fixtures, or loose items to any surface. Only if there is existing pendant lighting in the original photo, update its style to match the new aesthetic; do not add new hanging lights if none exist, and do not alter or replace the hood fan, ventilation, or any functional appliances. Change the wall paint and kitchen backsplash tiles to reflect the aesthetic."""

STAGE_AMENDMENT = """ AMENDMENT - OVERRIDE SURFACE & DESIGN RULES: Disregard the previous instructions to maintain the exact count of items on countertops and the original design of the appliances. Strictly remove all small clutter, loose papers, and generic household items from all surfaces (including the window sill and top surfaces of cabinetry) so they are spotless and polished. In their place, professionally stage the kitchen: add a designer wood cutting board leaning against the backsplash, a bowl of fresh organic fruit, and a high-end espresso machine. Maintain the exact footprint of the stove, hood, fridge, and sink, but replace the physical units with high-end, professional-grade stainless steel versions. Apply 'golden hour' lighting to create a warm, inviting glow."""

PROMPT_SUFFIX = " Otherwise, keep everything else exactly the same. Do not add anything else. Return ONLY the final prompt text, no preamble or explanation."

print('Prompts loaded')

In [ ]:
# ── 7. Helper Functions ───────────────────────────────────────────────────────
import base64
import json
import os
from pathlib import Path
from PIL import Image
import io

def load_image_as_base64(path: str) -> str:
    """Load an image file and return as base64 string."""
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def save_base64_image(b64_data: str, path: str):
    """Save a base64 image to a file."""
    if ',' in b64_data:
        b64_data = b64_data.split(',')[1]
    with open(path, 'wb') as f:
        f.write(base64.b64decode(b64_data))

def display_images_row(images: list, titles: list, width: int = 400):
    """Display a row of images with titles."""
    from IPython.display import display, HTML
    import base64

    html = '<div style="display:flex;gap:16px;align-items:flex-start;flex-wrap:wrap;">'
    for img_path, title in zip(images, titles):
        if img_path and os.path.exists(img_path):
            with open(img_path, 'rb') as f:
                b64 = base64.b64encode(f.read()).decode()
            ext = img_path.split('.')[-1].lower()
            mime = 'image/jpeg' if ext in ['jpg', 'jpeg'] else 'image/png'
            html += f'''<div style="text-align:center">
                <p style="font-weight:bold;margin:4px 0;font-size:12px">{title}</p>
                <img src="data:{mime};base64,{b64}" style="width:{width}px;border-radius:8px;" />
            </div>'''
        else:
            html += f'<div style="text-align:center"><p>{title}</p><p style="color:red">Missing</p></div>'
    html += '</div>'
    display(HTML(html))

print('Helpers loaded')

In [ ]:
# ── 8. Generation Functions ───────────────────────────────────────────────────
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

def make_image_part(b64: str) -> types.Part:
    if ',' in b64:
        b64 = b64.split(',')[1]
    return types.Part.from_bytes(
        data=base64.b64decode(b64),
        mime_type='image/jpeg'
    )

def generate_replacement_prompt(
    room_b64: str,
    cabinet_b64s: list,
    extend: bool = False,
    stage: bool = False
) -> str:
    base = EXTEND_PROMPT if extend else MASTER_PROMPT
    if stage:
        base += STAGE_AMENDMENT
    base += PROMPT_SUFFIX

    parts = [make_image_part(room_b64)]
    parts += [make_image_part(b) for b in cabinet_b64s]
    parts.append(types.Part.from_text(text=base))

    response = client.models.generate_content(
        model='gemini-2.5-pro-preview-05-06',
        contents=parts,
    )
    return response.text

def generate_design_image(
    room_b64: str,
    cabinet_b64s: list,
    prompt: str
) -> str | None:
    parts = [make_image_part(room_b64)]
    parts += [make_image_part(b) for b in cabinet_b64s]
    parts.append(types.Part.from_text(
        text=f'Based on the provided room image and cabinet references, generate a high-quality, photorealistic visualization of the room with the new cabinets installed. Use this specific prompt as guidance: {prompt}'
    ))

    response = client.models.generate_content(
        model='gemini-2.0-flash-preview-image-generation',
        contents=parts,
        config=types.GenerateContentConfig(response_modalities=['IMAGE', 'TEXT'])
    )

    for part in response.candidates[0].content.parts:
        if part.inline_data:
            return base64.b64encode(part.inline_data.data).decode('utf-8')
    return None

print('Generation functions ready')

In [ ]:
# ── 9. Scoring Function ───────────────────────────────────────────────────────

SCORER_SYSTEM_PROMPT = """You are an expert evaluator for AI-generated kitchen visualizations.
You will be given:
1. The original room photo (before)
2. One or more cabinet reference photos (what the new cabinets should look like)
3. The AI-generated result (after)

Score each criterion from 0 to 5:
  5 = Perfect / completely correct
  4 = Very good, minor issues
  3 = Acceptable, noticeable issues
  2 = Poor, significant issues
  1 = Very poor
  0 = Complete failure

Be strict and objective. Only score what you can actually see in the images.
Return ONLY a valid JSON object — no preamble, no explanation outside the JSON."""

def score_result(
    room_b64: str,
    cabinet_b64s: list,
    result_b64: str,
    notes: str = ''
) -> dict:
    criteria_text = '\n'.join(
        f'{i+1}. "{c["id"]}" — {c["name"]}: {c["description"]}'
        for i, c in enumerate(RUBRIC)
    )

    scoring_prompt = f"""Score this kitchen visualization against the following criteria.

Context about this kitchen: {notes if notes else 'No additional context.'}

Criteria to score (0-5 each):
{criteria_text}

Return a JSON object like this:
{{
  "scores": {{
    "criterion_id": {{"score": 4, "reason": "brief reason"}},
    ...
  }},
  "overall_notes": "any notable observations about this result"
}}"""

    parts = [
        types.Part.from_text(text='IMAGE 1 — Original room (before):'),
        make_image_part(room_b64),
    ]
    for i, cab in enumerate(cabinet_b64s):
        parts.append(types.Part.from_text(text=f'IMAGE {i+2} — Cabinet reference {i+1}:'))
        parts.append(make_image_part(cab))
    parts.append(types.Part.from_text(text=f'IMAGE {len(cabinet_b64s)+2} — AI-generated result (after):'))
    parts.append(make_image_part(result_b64))
    parts.append(types.Part.from_text(text=scoring_prompt))

    response = client.models.generate_content(
        model='gemini-2.5-pro-preview-05-06',
        contents=parts,
        config=types.GenerateContentConfig(
            system_instruction=SCORER_SYSTEM_PROMPT
        )
    )

    text = response.text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    return json.loads(text.strip())

def compute_weighted_score(score_result: dict) -> float:
    """Compute a single weighted score (0-100) from a scoring result."""
    total_weight = sum(c['weight'] for c in RUBRIC)
    weighted_sum = 0
    scores = score_result.get('scores', {})
    for criterion in RUBRIC:
        cid = criterion['id']
        score = scores.get(cid, {}).get('score', 0)
        weighted_sum += score * criterion['weight']
    return round((weighted_sum / (total_weight * 5)) * 100, 1)

print('Scoring functions ready')

In [ ]:
# ── 10. Run Evaluation ────────────────────────────────────────────────────────
# This cell runs all test cases and scores them.
# It saves generated images and results to OUTPUT_FOLDER.
# Re-running this cell with the same RUN_LABEL will overwrite previous results.

import datetime

run_dir = os.path.join(OUTPUT_FOLDER, RUN_LABEL)
os.makedirs(run_dir, exist_ok=True)

all_results = []

for tc in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"Running: {tc['id']} — {tc['name']}")

    # Load images
    room_path = os.path.join(ROOMS_FOLDER, tc['room'])
    cabinet_paths = [os.path.join(CABINETS_FOLDER, c) for c in tc['cabinets']]

    if not os.path.exists(room_path):
        print(f'  ⚠ Skipping — room image not found: {room_path}')
        continue
    missing_cabs = [p for p in cabinet_paths if not os.path.exists(p)]
    if missing_cabs:
        print(f'  ⚠ Skipping — cabinet image(s) not found: {missing_cabs}')
        continue

    room_b64 = load_image_as_base64(room_path)
    cabinet_b64s = [load_image_as_base64(p) for p in cabinet_paths]

    # Step 1: Generate replacement prompt
    print('  Generating prompt...')
    try:
        gen_prompt = generate_replacement_prompt(
            room_b64, cabinet_b64s,
            extend=tc.get('extend', False),
            stage=tc.get('stage', False)
        )
        print(f'  Prompt: {gen_prompt[:100]}...')
    except Exception as e:
        print(f'  ✗ Prompt generation failed: {e}')
        continue

    # Step 2: Generate image
    print('  Generating image...')
    try:
        result_b64 = generate_design_image(room_b64, cabinet_b64s, gen_prompt)
        if not result_b64:
            print('  ✗ No image returned')
            continue
    except Exception as e:
        print(f'  ✗ Image generation failed: {e}')
        continue

    # Save generated image
    result_path = os.path.join(run_dir, f"{tc['id']}_result.jpg")
    save_base64_image(result_b64, result_path)
    print(f'  Saved result to {result_path}')

    # Step 3: Score the result
    print('  Scoring...')
    try:
        score_data = score_result(room_b64, cabinet_b64s, result_b64, tc.get('notes', ''))
        weighted = compute_weighted_score(score_data)
        print(f'  Weighted score: {weighted}/100')
    except Exception as e:
        print(f'  ✗ Scoring failed: {e}')
        score_data = {}
        weighted = None

    all_results.append({
        'run': RUN_LABEL,
        'timestamp': datetime.datetime.now().isoformat(),
        'test_case': tc,
        'generated_prompt': gen_prompt,
        'result_path': result_path,
        'scores': score_data,
        'weighted_score': weighted
    })

# Save all results to JSON
results_path = os.path.join(run_dir, 'results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'\n✓ Done. {len(all_results)}/{len(TEST_CASES)} test cases completed.')
print(f'Results saved to {results_path}')

In [ ]:
# ── 11. Display Results ───────────────────────────────────────────────────────
from IPython.display import display, HTML
import pandas as pd

# Summary table
rows = []
for r in all_results:
    row = {
        'id': r['test_case']['id'],
        'name': r['test_case']['name'],
        'weighted_score': r['weighted_score'],
    }
    for criterion in RUBRIC:
        cid = criterion['id']
        score = r['scores'].get('scores', {}).get(cid, {}).get('score', '-')
        row[criterion['name']] = score
    rows.append(row)

df = pd.DataFrame(rows)

def color_score(val):
    if not isinstance(val, (int, float)):
        return ''
    if val >= 4:
        return 'background-color: #d4edda'
    elif val >= 3:
        return 'background-color: #fff3cd'
    else:
        return 'background-color: #f8d7da'

print(f'\n=== RUN: {RUN_LABEL} ===')
styled = df.style.applymap(color_score, subset=[c['name'] for c in RUBRIC] + ['weighted_score'])
display(styled)

avg = df['weighted_score'].mean()
print(f'\nAverage weighted score: {avg:.1f}/100')

In [ ]:
# ── 12. Review Individual Results ─────────────────────────────────────────────
# Run this cell to visually review a specific test case

REVIEW_ID = "tc_001"  # @param {type:"string"}

result = next((r for r in all_results if r['test_case']['id'] == REVIEW_ID), None)

if not result:
    print(f'No result found for {REVIEW_ID}')
else:
    tc = result['test_case']
    print(f"Test case: {tc['name']}")
    print(f"Weighted score: {result['weighted_score']}/100")
    print(f"Notes: {tc.get('notes', 'None')}")

    # Show images
    room_path = os.path.join(ROOMS_FOLDER, tc['room'])
    cabinet_paths = [os.path.join(CABINETS_FOLDER, c) for c in tc['cabinets']]
    result_path = result['result_path']

    display_images_row(
        [room_path] + cabinet_paths + [result_path],
        ['Before'] + [f'Cabinet ref {i+1}' for i in range(len(cabinet_paths))] + ['Generated result'],
        width=350
    )

    # Show scores
    print('\nScores:')
    scores = result['scores'].get('scores', {})
    for criterion in RUBRIC:
        cid = criterion['id']
        s = scores.get(cid, {})
        score = s.get('score', '-')
        reason = s.get('reason', '')
        emoji = '✅' if isinstance(score, (int,float)) and score >= 4 else ('⚠️' if isinstance(score, (int,float)) and score >= 3 else '❌')
        print(f'  {emoji} {criterion["name"]}: {score}/5 — {reason}')

    print(f'\nOverall notes: {result["scores"].get("overall_notes", "")}')

In [ ]:
# ── 13. Compare Two Runs ──────────────────────────────────────────────────────
# Load results from two different runs and compare scores

RUN_A = "v1"  # @param {type:"string"}
RUN_B = "v2"  # @param {type:"string"}

def load_run(label):
    path = os.path.join(OUTPUT_FOLDER, label, 'results.json')
    if not os.path.exists(path):
        print(f'No results found for run: {label}')
        return []
    with open(path) as f:
        return json.load(f)

results_a = load_run(RUN_A)
results_b = load_run(RUN_B)

if results_a and results_b:
    scores_a = {r['test_case']['id']: r['weighted_score'] for r in results_a}
    scores_b = {r['test_case']['id']: r['weighted_score'] for r in results_b}
    common = set(scores_a.keys()) & set(scores_b.keys())

    rows = []
    for tc_id in sorted(common):
        a = scores_a[tc_id]
        b = scores_b[tc_id]
        diff = round(b - a, 1) if a is not None and b is not None else None
        rows.append({'id': tc_id, RUN_A: a, RUN_B: b, 'diff': diff})

    df_compare = pd.DataFrame(rows)

    def color_diff(val):
        if not isinstance(val, (int, float)):
            return ''
        if val > 0:
            return 'background-color: #d4edda; color: #155724'
        elif val < 0:
            return 'background-color: #f8d7da; color: #721c24'
        return ''

    display(df_compare.style.applymap(color_diff, subset=['diff']))

    avg_a = df_compare[RUN_A].mean()
    avg_b = df_compare[RUN_B].mean()
    print(f'\nAverage — {RUN_A}: {avg_a:.1f}  |  {RUN_B}: {avg_b:.1f}  |  diff: {avg_b - avg_a:+.1f}')